# 01. Dataset overview

Audit bản dataset candidate trước khi nhóm quyết định có dùng cho phân loại CWE hay không.

Notebook này chỉ khảo sát. Không xóa dòng, không chọn ngưỡng CWE, không lấy CWE đầu tiên làm nhãn, không chốt chiến lược split.

Mọi số liệu phải tính từ file đã tải. Không điền số từ paper. Logic dùng lại được thì đưa vào `src/data/`, không viết pipeline nghiên cứu ở đây.


## 0. Setup

Chạy Jupyter từ thư mục gốc repository sau khi `pip install -r requirements.txt`.

Dataset candidate hiện tại là DiverseVul. File config: `configs/data/diversevul.yaml`. Đặt dữ liệu vào `data/raw/diversevul/` và ghi `version` trong config. Không commit dữ liệu thô.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    raise RuntimeError("Run this notebook from the repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import load_config
from src.utils.paths import report_dir, resolve_from_root

config = load_config(ROOT / "configs" / "data" / "diversevul.yaml")
dataset = config["dataset"]
raw_dir = resolve_from_root(dataset["local_raw_dir"])
table_dir = report_dir("dataset", "tables")
figure_dir = report_dir("dataset", "figures")

print("dataset:", dataset["name"])
print("status:", dataset["status"])
print("version:", dataset["version"])
print("raw_dir:", raw_dir)
print("raw_dir_exists:", raw_dir.exists())
print("files:", sorted(path.name for path in raw_dir.glob("*")) if raw_dir.exists() else [])
print("Next: load the files with src.data.loader.load_dataset. That loader is still a skeleton.")


## A. Dataset overview

Tính từ dữ liệu thật:

- Dataset có bao nhiêu records?
- Bao nhiêu vulnerable?
- Bao nhiêu non-vulnerable?
- Tỷ lệ vulnerable là bao nhiêu?
- Có những columns hoặc fields nào?
- Missing values của từng field?
- Source code có missing không?
- Project metadata có missing không?
- Commit metadata có missing không?
- CWE metadata có missing không?

Ghi schema thật vào config (`code_field`, `label_field`, `cwe_field`, `project_field`, `commit_field`, `hash_field`, `timestamp_field`) sau khi đã kiểm tra. Không đoán tên cột.


In [ ]:
# TODO: implement src.data.loader.load_dataset, then replace this stop with a frame summary.
# Do not print the full frame.
if not raw_dir.exists() or not any(raw_dir.iterdir()):
    print("STOPPED: no dataset files in", raw_dir)
    print("Download DiverseVul manually, set dataset.version, then rerun from the top.")
else:
    raise NotImplementedError(
        "Files are present. Implement load_dataset and compute record counts, "
        "class balance, columns, and missing values here."
    )


## B. CWE analysis

Chỉ phân tích CWE có thật trong dữ liệu. Hàm dự kiến: `src.data.cwe.cwe_distribution` và `src.data.cwe.threshold_summary`.

Cần tính:

- số CWE unique
- số vulnerable functions có CWE
- số vulnerable functions không có CWE
- số samples của từng CWE
- percentage của từng CWE
- Top 10 CWE
- Top 20 CWE
- dạng long-tail

Xuất `reports/dataset/tables/cwe_distribution.csv`:

| CWE | sample_count | percentage | project_count |
| --- | --- | --- | --- |

Với từng ngưỡng `>= 20`, `>= 50`, `>= 100`, `>= 200`, báo cáo số CWE còn lại, số vulnerable functions còn lại, và phần trăm dữ liệu giữ lại. Không chọn ngưỡng.


In [ ]:
print("TODO: call src.data.cwe.cwe_distribution(records)")
print("TODO: call src.data.cwe.threshold_summary(distribution)")
print("Write reports/dataset/tables/cwe_distribution.csv")
print("Thresholds to report, without selecting one: 20, 50, 100, 200")


## C. Multi-CWE analysis

Hàm dự kiến: `src.data.cwe.parse_cwe_labels` và `src.data.cwe.multi_cwe_summary`.

Báo cáo:

- số function có đúng 1 CWE
- số function có từ 2 CWE trở lên
- tỷ lệ multi-CWE
- phân bố số CWE trên một sample
- vài ví dụ multi-CWE

Không lấy CWE đầu tiên làm ground-truth.


In [ ]:
print("TODO: call src.data.cwe.multi_cwe_summary(records)")
print("Show a handful of multi-CWE examples, not the whole dataset.")


## D. Project analysis

Xuất:

- `reports/dataset/tables/project_distribution.csv` với `project`, `sample_count`, `vulnerable_count`, `cwe_count`
- `reports/dataset/tables/cwe_project_distribution.csv` với `CWE`, `sample_count`, `project_count`

Cần trả lời:

- số project
- số samples trên mỗi project
- số vulnerable samples trên mỗi project
- số CWE trên mỗi project
- số project trên mỗi CWE
- random split có khả thi không
- project-wise split có khả thi không
- metadata thời gian có đủ để làm chronological split không

Một CWE nhiều sample nhưng chỉ nằm ở một project là rủi ro cho project-wise split. Ghi các trường hợp đó. Chưa chốt split.


In [ ]:
print("TODO: build the project table and the CWE-by-project table.")
print("Flag CWE values whose project_count is 1.")
print("Record whether a timestamp field exists and how many values are missing.")


## E. Duplicate analysis

Chỉ báo cáo, không xóa. Hàm dự kiến nằm trong `src.data.deduplication`.

Kiểm tra:

- trùng source code chính xác
- trùng hash nếu dataset có cột hash
- cùng source code nhưng khác nhãn
- cùng function xuất hiện ở nhiều records
- trùng sau khi chuẩn hóa line ending và whitespace
- trùng sau khi bỏ comment, chỉ khi cách bỏ comment an toàn với C/C++


In [ ]:
print("TODO: exact_source_duplicates, hash_duplicates, conflicting_label_duplicates")
print("TODO: normalized_source_duplicates via src.data.preprocessing")
print("Leave every original row in place.")


## F. Source code quality

Kiểm tra function rỗng, function cực ngắn, function cực dài, phân bố số dòng, phân bố độ dài ký tự hoặc token, và mẫu nhìn như malformed nếu phát hiện được.

Lấy ngẫu nhiên một ít function vulnerable để xem thủ công. Không in toàn bộ dataset.


In [ ]:
print("TODO: summarize line counts and character lengths.")
print("TODO: print a small random sample of vulnerable functions for manual inspection.")


## G. Graph readiness

Chưa sinh CPG cho toàn bộ dữ liệu. CPG và Joern vẫn là candidate.

Chỉ cần kết luận source code có đủ để bước trích graph chạy thử hay không. Nếu thuận tiện, ghi một manifest khoảng 20 đến 50 function (ID và đường dẫn), để pha sau gọi Joern. Không chạy Joern trong notebook này.


In [ ]:
print("TODO: count samples with missing or empty source code.")
print("TODO: optionally write a 20-50 function manifest under data/interim and record the path.")
print("Do not export CPG files for the full dataset.")


## Kết luận cần ghi vào báo cáo

Sau khi các mục trên có số, cập nhật `reports/dataset/dataset_eda.md`. Mỗi kết luận phải dẫn từ bảng vừa tính:

1. DiverseVul có phù hợp với CWE classification hay không.
2. Có bao nhiêu CWE thực sự usable, kèm ngưỡng đang được xem xét chứ chưa chốt.
3. Class imbalance nghiêm trọng tới mức nào.
4. Multi-CWE có phải vấn đề đáng kể không.
5. Duplicate có nghiêm trọng không.
6. Project-wise split có khả thi không.
7. Source code có đủ điều kiện để thử graph extraction không.
8. Nhóm cần quyết định gì tiếp theo.

Không tuyên bố DiverseVul là dataset cuối cùng nếu bằng chứng chưa đủ. Nếu số khác paper, ghi số của file đang dùng và phiên bản dữ liệu.


In [ ]:
print("Report template:", report_dir("dataset", "dataset_eda.md"))
print("Tables:", table_dir)
print("Figures:", figure_dir)
print("Expected tables:")
print("- cwe_distribution.csv")
print("- project_distribution.csv")
print("- cwe_project_distribution.csv")
